# `drone_env.ipynb` : environnement Gymnasium autour du RNN identifié

Ce notebook définit tout ce dont `main_and_figures.ipynb` a besoin :

- `RNNPlant` : ton `nn.RNN` réimplémenté en numpy, pas à pas ;
- `mission(t)` ;
- `ObservationBuilder` : la partie à recopier dans `observation()` de `rl_controller.m` ;
- `DroneEnv` : l'environnement ;
- `check_like_matlab()` : le check de `EINT_RL_eval.m`, rejoué en Python.

Depuis `main_and_figures.ipynb`, on l'importe avec `%run -i drone_env.ipynb`. Avec `RUN_ENV_TESTS = False` défini
avant, les tests en bas de ce notebook ne sont pas relancés.

Les endroits à régler sont marqués **`# >>> IMPORTANT`** dans le code.

### Avant tout : exporter le RNN depuis `checkdata.ipynb`

Ajoute cette cellule à la fin de `checkdata.ipynb`, après l'entraînement, et exécute-la. Elle écrit
`rnn_plant.npz` (à mettre à côté de ce notebook) avec les poids **et toute la chaîne de normalisation**.

```python
sd = {k: v.detach().cpu().numpy() for k, v in model.state_dict().items()}
np.savez("rnn_plant.npz",
         W_ih=sd["rnn.weight_ih_l0"], W_hh=sd["rnn.weight_hh_l0"],
         b_ih=sd["rnn.bias_ih_l0"],   b_hh=sd["rnn.bias_hh_l0"],
         W_out=sd["output.weight"],   b_out=sd["output.bias"],
         x_mean=float(x_mean), x_std=float(x_std), y_mean=float(y_mean), y_std=float(y_std),
         mm_scale=np.asarray(sc.scale_, float), mm_min=np.asarray(sc.min_, float))
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from scipy.io import loadmat

TS         = 0.05          # imposé (export_policy.py, EINT_RL_eval.m)
N_STEPS    = 200           # 10 s de mission
G          = 9.81
PLANT_PATH = "rnn_plant.npz"

## 1. Mission (Fig. 1)

Le drone tient 0 m sur la plateforme, monte à 0,5 m, puis descend se poser à −0,25 m.

- Le temps est `t = k*TS`, jamais `t += TS`.
- Les seuils sont décalés de `EPS_T` pour que Python et Simulink tombent du même côté d'une marche.
- **Avec les valeurs par défaut, cette fonction doit être identique à `mission()` en MATLAB.** Les paramètres
  ne servent qu'à la randomisation de l'entraînement.

In [ ]:
# >>> IMPORTANT : la mission. Si tu changes ces valeurs, change aussi mission() dans rl_controller.m
T_UP, T_DOWN = 1.0, 6.0      # instants des marches [s]
Z_HIGH       = 0.5           # palier haut [m]
Z_GROUND     = -0.25         # sol [m]
EPS_T        = 1e-6

def mission(t, t_up=T_UP, t_down=T_DOWN, z_high=Z_HIGH):
    if t < t_up - EPS_T:
        return 0.0
    if t < t_down - EPS_T:
        return float(z_high)
    return Z_GROUND

## 2. Le modèle : ton RNN, pas à pas

### Ce qu'on reproduit

Ton réseau est `nn.RNN(1, 64)` (cellule tanh), puis `nn.Linear(64, 1)`, avec trois transformations autour :

1. **entrée** : $u_n = (u - \bar x)/\sigma_x$ ;
2. **sortie** : $y_{mm} = y_n\,\sigma_y + \bar y$ ;
3. **puis** l'inverse du `MinMaxScaler`, $a_z = (y_{mm} - \text{min\_}[j]) / \text{scale\_}[j]$.

La cellule s'écrit $h_j = \tanh(W_{ih}u_{n,j} + b_{ih} + W_{hh}h_{j-1} + b_{hh})$, avec $h_{-1} = 0$
(défaut PyTorch). On la recode en numpy : c'est plus rapide pas à pas, et on peut « regarder un pas en avant »
sans modifier l'état.

### Point 1 : le `MinMaxScaler` est appliqué par colonne, donc par instant

`sc.fit_transform(df_out)` sur un tableau (signaux × 2000 instants) normalise **chaque instant séparément**.
`scale_[j]` et `min_[j]` dépendent donc de l'instant `j` : ton modèle est indexé par le temps.

Pendant les 2 premières secondes, `u = 0` dans tous les signaux. La plage de az y est minuscule (0,33 contre
environ 40 ensuite), et le RNN n'a jamais vu de commande non nulle avant 2 s. Si on démarrait l'épisode à
l'instant 0 du RNN, le drone ne réagirait à rien pendant 2 s.

**Solution retenue (sans réentraîner)** : un **préchauffage** de 40 pas avec `u = 0` (de 0 à 2 s), puis la
mission démarre à l'instant `j = 40` du RNN. C'est exactement la situation des données : drone en chute libre
(az ≈ −9,8, ce que fait aussi le simulateur à t = 0 avec `u = 0`), puis première commande à 2 s.
La mission utilise les instants 40 à 240, tous couverts par le scaler.

Il reste une variation de `scale_[j]` de quelques % d'un instant à l'autre, qui agit comme un petit bruit de
gain. Réentraîner avec un scaler **global** (`sc.fit(df_out.to_numpy().reshape(-1, 1))`) serait plus propre.
`RNNPlant` gère les deux cas : si `mm_scale` est un scalaire, il s'applique à tous les instants.

### Point 2 : l'alignement temporel (pourquoi on « regarde un pas en avant »)

Dans tes données, `scale_` à 2,00 s est encore de 0,31 : l'az à l'instant $t_j$ ne dépend **pas** de $u_j$,
seulement des commandes passées (pas de transmission directe). Le RNN l'a appris.

Sur un pas $[t_k, t_{k+1}]$, avec `u_k` maintenu (ZOH), on veut donc :

- $a_z(t_k)$ = sortie après avoir consommé $u_k$ à l'instant $j$ (elle n'en dépend quasiment pas) ;
- $a_z(t_{k+1})$ = sortie à l'instant $j+1$ avec $u_k$ encore appliqué : un pas « en avant », **sans** valider
  l'état, puisque le vrai $u_{k+1}$ n'est pas encore connu.

On intègre ensuite avec la moyenne des deux (trapèzes). Sans cela, l'effet de chaque commande arriverait avec un
demi-pas de retard par rapport au simulateur. C'est aussi ce qui donne `az_prev` = $a_z(t_k)$, comme le
`UnitDelay` de Simulink.

In [ ]:
class RNNPlant:
    # Ton nn.RNN(1, H) + Linear(H, 1), avec la normalisation de checkdata, pas à pas
    def __init__(self, path=PLANT_PATH, warmup=40):
        d = np.load(path)
        self.W_ih = d["W_ih"].astype(float)[:, 0]                 # (H,)
        self.W_hh = d["W_hh"].astype(float)                       # (H, H)
        self.b = (d["b_ih"] + d["b_hh"]).astype(float)            # (H,)
        self.W_out = d["W_out"].astype(float)[0]                  # (H,)
        self.b_out = float(np.ravel(d["b_out"])[0])
        self.x_mean, self.x_std = float(d["x_mean"]), float(d["x_std"])
        self.y_mean, self.y_std = float(d["y_mean"]), float(d["y_std"])
        self.mm_scale = np.atleast_1d(d["mm_scale"]).astype(float)   # par instant (ou scalaire si scaler global)
        self.mm_min = np.atleast_1d(d["mm_min"]).astype(float)
        self.warmup = warmup                                      # >>> IMPORTANT : 40 pas = 2 s de u = 0 dans les données

    def _cell(self, h, u):
        return np.tanh(self.W_ih * ((u - self.x_mean) / self.x_std) + self.b + self.W_hh @ h)

    def _az(self, h, j):
        y_mm = (self.W_out @ h + self.b_out) * self.y_std + self.y_mean
        jj = min(j, len(self.mm_scale) - 1)
        return (y_mm - self.mm_min[jj]) / self.mm_scale[jj]        # inverse du MinMaxScaler à l'instant j

    def reset(self):
        self.h = np.zeros_like(self.b)
        for _ in range(self.warmup):                              # 0 -> 2 s à u = 0, comme dans les données
            self.h = self._cell(self.h, 0.0)
        self.j = self.warmup - 1                                  # dernier instant consommé

    def step(self, u):
        # u maintenu sur [t_k, t_k+1]. Renvoie (az(t_k), az(t_k+1))
        self.j += 1
        self.h = self._cell(self.h, u)                            # on valide l'instant j avec u_k
        az_now = self._az(self.h, self.j)
        az_next = self._az(self._cell(self.h, u), self.j + 1)     # un pas en avant, état NON validé
        return az_now, az_next

    def free_response(self, u_seq):
        # simulation libre (sans contrôleur) : z, az
        self.reset(); z = vz = 0.0; Z, A = [], []
        for u in u_seq:
            a0, a1 = self.step(float(u))
            vz_new = vz + 0.5 * (a0 + a1) * TS; z += 0.5 * (vz + vz_new) * TS; vz = vz_new
            Z.append(z); A.append(a0)
        return np.array(Z), np.array(A)

    def hover_command(self):
        # u tel que az ≈ 0 en régime établi (dichotomie)
        def az_ss(u):
            return self.free_response(np.full(30, u))[1][-10:].mean()
        lo, hi = -1.0, 1.0
        if not az_ss(lo) < 0 < az_ss(hi):
            raise ValueError("az ne change pas de signe sur [-1, 1] : vérifie la normalisation")
        for _ in range(25):
            mid = 0.5 * (lo + hi)
            lo, hi = (mid, hi) if az_ss(mid) < 0 else (lo, mid)
        return 0.5 * (lo + hi)

## 3. L'observation : la partie à recopier dans `rl_controller.m`

Elle ne dépend **que** des mesures `m = {t, z, vz, az_prev, z_ref, u_prev}`, plus une mémoire (l'intégrale).
Il n'y a pas d'état du RNN dedans, ni de `VecNormalize` : rien de cela n'existe côté MATLAB.

| # | Composante | Échelle | Rôle |
|---|---|---|---|
| 1 | `e = z_ref − z` | /0,5 | erreur de suivi ; invariance par translation |
| 2 | `vz` | /1 | sans elle, double intégrateur non observé, donc oscillations |
| 3 | `az_prev` | /10 | état des moteurs (dynamique d'environ 0,1 à 0,3 s) ; vaut −9,8 en chute libre, 0 en vol stationnaire |
| 4 | `u_prev` | /1 | rend le problème markovien, puisqu'on pénalise Δu |
| 5 | `z_ref` | /0,5 | indique la phase (atterrissage ou non) |
| 6 | `ie = ∫e dt`, bornée ±`IE_MAX` | /`IE_MAX` | compense un écart de poussée d'équilibre RNN/simulateur (erreur statique) |

`__call__` doit être appelé **une seule fois par pas** (un dans `reset`, un par `step`). C'est ce qui rend
`self.ie` identique à la `persistent ie` de MATLAB, appelée une fois par pas Simulink.

In [ ]:
# >>> IMPORTANT : l'observation. Toute modification doit être reportée dans observation() de rl_controller.m
IE_MAX    = 0.2
E_INT     = 0.3               # intégration conditionnelle (anti-windup), comme dans rl_controller.m
OBS_SCALE = np.array([1/0.5, 1/1.0, 1/10.0, 1.0, 1/0.5, 1/IE_MAX])

class ObservationBuilder:
    def reset(self):
        self.ie = 0.0
        self.zref_prev = None
    def __call__(self, m):
        e = m["z_ref"] - m["z"]
        if self.zref_prev is None:
            self.zref_prev = m["z_ref"]
        if abs(m["z_ref"] - self.zref_prev) > 0.1:   # nouvelle marche : on oublie l'erreur du palier précédent
            self.ie = 0.0
        self.zref_prev = m["z_ref"]
        if abs(e) < E_INT:
            self.ie = min(max(self.ie + e * TS, -IE_MAX), IE_MAX)
        raw = np.array([e, m["vz"], m["az_prev"], m["u_prev"], m["z_ref"], self.ie])
        return (raw * OBS_SCALE).astype(np.float32)

## 4. L'environnement

### Chronologie d'un pas, alignée sur `EINT_RL_closed_loop.slx`

D'après le câblage du modèle Simulink :

- `z` et `vz` passent par un ZOH ;
- `az_prev` et `u_prev` sortent de `UnitDelay` à condition initiale **0** ;
- le même `u` va aux 4 moteurs.

```
reset : RNN préchauffé (2 s, u=0) ; z = vz = 0 ; az_prev = u_prev = 0      → m_0, obs_0
step(u_k) : (az_k, az_k+1) = RNN(u_k) ; trapèzes → vz_k+1, z_k+1
            az_prev ← az_k ; u_prev ← u_k ; k ← k+1                       → m_k+1, obs_k+1, r_k
```

### Action centrée sur la poussée d'équilibre (`# >>> IMPORTANT : ACTION`)

Au début de l'entraînement, la politique SAC sort une action proche du centre de l'intervalle. Avec `[-1, 1]`,
cela donne `u ≈ 0`, donc la chute libre (az = −9,8) : l'agent dépense une bonne partie de son budget à
découvrir qu'il faut `u ≈ 0,38` pour simplement tenir. Dans mon essai à 100k pas avec `[-1, 1]`, la politique
restait moins bonne que le PID, avec une erreur statique d'environ 18 cm.

On prend donc `action_space = Box(U_HOVER − 0,6 ; U_HOVER + 0,6)` ≈ `[−0,22 ; 0,98]` : la politique initiale
vole déjà en stationnaire. Ce n'est **pas** une triche vis-à-vis de MATLAB : `export_for_matlab` enregistre
`action_low` et `action_high`, et `rl_controller.m` remet la sortie du réseau dans cet intervalle. C'est une
restriction de la commande, qui reste dans `[−1, 1]`. `check_env` affichera un avertissement « Box non
symétrique » : SAC remet lui-même l'action à l'échelle, on peut l'ignorer.

### Pas de sol dans l'env

Dans ton essai Simulink (`nouveau_test_rnn.csv`), avec `u = 0` pendant 5 s, az reste à −9,8 m/s² : le simulateur
**n'a pas de sol**. « Se poser » veut donc dire suivre −0,25 m sans passer en dessous ; c'est le `min z` affiché
par `EINT_RL_eval`. L'env ne bloque donc pas `z` à −0,25 (sinon la politique apprendrait à « s'appuyer » sur un sol
qui n'existe pas), mais **pénalise** fortement le passage sous le sol.

### Récompense (`# >>> IMPORTANT : REWARD`)

$$r_k = -\Big(\tfrac{e}{0{,}5}\Big)^2 - w_{|e|}\tfrac{|e|}{0{,}5} - w_{\Delta u}(u_k-u_{k-1})^2 - w_v v_z^2 - w_{sol}\,\tfrac{\max(0,\ z_{sol}-z)}{0{,}05}$$

| Terme | Pourquoi |
|---|---|
| $e^2$ | suivi, et pénalise fort les grandes erreurs (marches) |
| $\lvert e\rvert$ | e² seul tolère une erreur statique : 5 cm ne coûtent que 0,01. Le terme linéaire garde un gradient près de 0 |
| $\Delta u^2$ | évite le bang-bang ±1. Ton RNN a appris sur des paliers d'environ 1 s ; une commande qui change à chaque pas sort de son domaine de validité, et passe mal sur le simulateur |
| $v_z^2$ | amortit les dépassements |
| sous le sol | « crash » ; linéaire en profondeur (5 cm = 1 unité) |
| **pas** de $u^2$ | voler exige $u \approx$ U_HOVER ≈ 0,38 ≠ 0 : pénaliser $u^2$ créerait une erreur statique |

**Divergence** (`z > Z_MAX` ou `z < Z_MIN`) : fin d'épisode avec une pénalité égale au pire coût des pas restants.
Toutes les récompenses étant négatives, une mort anticipée moins chère que de continuer apprendrait à
l'agent à se « suicider ».

La récompense est calculée sur l'état **vrai** ; seule l'observation est bruitée.

### Robustesse (`# >>> IMPORTANT : RANDOMISATION`, uniquement si `randomize=True`)

La politique sera jugée sur `UAV_model.slx`, pas sur ton RNN. On l'entraîne donc sur une famille de modèles
autour du RNN :

| Tirage par épisode | Écart couvert |
|---|---|
| biais $a_z + U(-1;\,1)$ m/s² | erreur de poussée d'équilibre → appris avec l'intégrale |
| bruit de mesure sur `z`, `vz`, `az_prev` | `g_noise` du simulateur |

Comme ton pote : pas de gain, pas de retard, mission toujours nominale.

`NOISE_STD` est à **calibrer** : lance `EINT_system_simulation.slx` avec `g_noise = 0.1` et `u` constant,
et mesure l'écart-type de `z`, `vz` et `az` autour de leur tendance. Ton essai montre environ ±0,1 m/s² sur az.

**L'env passé à `export_for_matlab` doit avoir `noise=0, randomize=False`.** `reset(seed=0)` y donne
exactement la mission de la Fig. 1 sur le RNN nominal.

In [ ]:
# >>> IMPORTANT : REWARD (poids) : reprise de la récompense de ton pote
E_TRACK  = 0.3               # cloche large (au lieu de 0.15)
W_PREC   = 1.0               # poids de la cloche étroite
E_PREC   = 0.03              # 3 cm : à 2 cm on perd déjà 36 % du bonus
W_V      = 0.02              # au lieu de 0.05 : moins de freinage, convergence plus rapide             # amortissement : - W_V * min(vz², V2_MAX)
V2_MAX   = 4.0
W_DU     = 0.2                # douceur de la commande : - W_DU * Δu²
W_SOL    = 1.0                # pénalité sous Z_SAFE (mets 0 pour avoir exactement la récompense de ton pote)
SOL_MAX  = 5.0
Z_SAFE   = Z_GROUND + 0.02
Z_MAX, Z_MIN = 2.0, -1.0      # au-delà : divergence -> fin d'épisode
R_WORST  = 6.0                # majorant du coût d'un pas (pénalité de divergence)

W_SOL   = 0.0      # ou mettre Z_SAFE sous -0.25
R_WORST = 0.0      # plus de pénalité de mort géante
Z_MAX, Z_MIN = 2.0, -2.0
U_HALF  = 0.4      # ≈ [0, 0.8] comme ton pote

# >>> IMPORTANT : RANDOMISATION (comme ton pote : un biais sur az par épisode) et bruit de mesure (pour noise = 1)
BIAS_MAX    = 1.0             # m/s²
NOISE_STD   = dict(z=0.01, vz=0.03, az_prev=0.1)     # à calibrer sur le simulateur

# >>> IMPORTANT : DÉFAUTS RÉELS (valeurs par défaut = aucun défaut, pour env_eval et l'export)
DEFAULT_DEFECTS = dict(
    delay_max   = 0,      # retard commande : tiré dans {0, ..., delay_max} pas par épisode (1 pas = 50 ms)
    p_loss_cmd  = 0.0,    # proba de perdre un paquet commande -> le moteur garde la dernière commande reçue
    p_loss_meas = 0.0,    # proba de perdre un paquet mesure  -> le contrôleur réutilise l'ancienne mesure
    gust_std    = 0.0,    # rafales verticales : écart-type de l'accélération (m/s²), processus corrélé
    gust_tau    = 1.0,    # temps de corrélation des rafales (s)
    ground_k    = 0.0,    # effet de sol : surplus de poussée relatif au ras du sol (ex. 0.15 = +15 %)
    ground_h    = 0.10,   # hauteur caractéristique de l'effet de sol (m)
    gain_drift  = 0.0,    # perte de poussée relative sur l'épisode (batterie), ex. 0.05 = -5 % en 10 s
    quant_bits  = 0,      # quantification de la commande sur [-1, 1] (bits) ; 0 = pas de quantification
    vz_from_z   = False,  # vz non mesurée : estimée par différence finie de z (bruité)
)

class DroneEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, noise=0.0, randomize=False, plant_path=PLANT_PATH,defects=None):
        super().__init__()
        self.D={**DEFAULT_DEFECTS,**(defects or {})}
        self.plant = RNNPlant(plant_path)                 # une instance par env (le RNN a un état)
        self.noise, self.randomize = noise, randomize
        self.obs_builder = ObservationBuilder()
        u0 = round(self.plant.hover_command(), 3)         # politique initiale (sortie ≈ 0) = vol stationnaire
        self.u_lo, self.u_hi = max(-1.0, u0 - U_HALF), min(1.0, u0 + U_HALF)
        self.action_space = spaces.Box(self.u_lo, self.u_hi, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(len(OBS_SCALE),), dtype=np.float32)

    # ---- tirage des paramètres de l'épisode --------------------------------------
    def _sample_episode(self):
        self.bias = self.np_random.uniform(-BIAS_MAX, BIAS_MAX) if self.randomize else 0.0

    def _perturb(self, az):
        return az + self.bias

    # ---- mesures du contrôleur (ce que voit rl_controller.m) -------------------------
    def _measure(self):
        t = self.k * TS                                   # jamais t += TS
        z, vz, az_prev = self.z, self.vz, self.az_prev
        if self.noise > 0:
            rng = self.np_random
            z += self.noise * NOISE_STD["z"] * rng.standard_normal()
            vz += self.noise * NOISE_STD["vz"] * rng.standard_normal()
            az_prev += self.noise * NOISE_STD["az_prev"] * rng.standard_normal()
        m = dict(t=t, z=z, vz=vz, az_prev=az_prev, z_ref=mission(t), u_prev=self.u_prev)
        # --- défauts côté capteurs : vz estimée, perte de paquet mesure
        D, rng = self.D, self.np_random
        if D["vz_from_z"]:
            m["vz"] = (m["z"] - self.z_meas_prev) / TS
            self.z_meas_prev = m["z"]
        if self.m_last is not None and rng.random() < D["p_loss_meas"]:
            m = dict(self.m_last, t=t, z_ref=m["z_ref"], u_prev=self.u_prev)   # mesure périmée
        self.m_last = m
        return m

    # ---- API Gymnasium --------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._sample_episode()
        self.plant.reset()                                # préchauffage 2 s à u = 0
        self.k = 0
        self.z, self.vz = 0.0, 0.0
        self.az_prev, self.u_prev = 0.0, 0.0
                # --- état des défauts réels
        D, rng = self.D, self.np_random
        self.delay = int(rng.integers(0, D["delay_max"] + 1)) if D["delay_max"] else 0
        self.u_queue = [0.0] * self.delay      # commandes "en vol" sur le réseau
        self.u_motor = 0.0                     # commande réellement appliquée par les moteurs
        self.gust = 0.0
        self.m_last = None                     # dernière mesure reçue (pertes de paquets)
        self.z_meas_prev = 0.0                 # pour estimer vz à partir de z              # conditions initiales des UnitDelay
        self.obs_builder.reset()
        m = self._measure()
        return self.obs_builder(m), {"m": m}

    def step(self, action):
        u = float(np.clip(np.asarray(action, dtype=np.float64).reshape(-1)[0], self.u_lo, self.u_hi))
        u_old = self.u_prev

                # --- défauts côté actionneur : quantification, retard, perte de paquet commande
        D, rng = self.D, self.np_random
        if D["quant_bits"]:
            q = 2.0 / 2 ** D["quant_bits"]
            u_sent = round(u / q) * q
        else:
            u_sent = u
        self.u_queue.append(u_sent)
        u_arrived = self.u_queue.pop(0)
        if rng.random() >= D["p_loss_cmd"]:    # paquet reçu : sinon le moteur garde l'ancienne commande
            self.u_motor = u_arrived

        az_now, az_next = self.plant.step(self.u_motor)
        az_now, az_next = self._perturb(az_now), self._perturb(az_next)

        # --- défauts côté physique : batterie, rafales, effet de sol
        gain = 1.0 - D["gain_drift"] * self.k / N_STEPS
        a = np.exp(-TS / D["gust_tau"])
        self.gust = a * self.gust + D["gust_std"] * np.sqrt(1 - a * a) * rng.standard_normal()
        thrust_now, thrust_next = az_now + G, az_next + G
        ge = 1.0 + D["ground_k"] * np.exp(-max(self.z - Z_GROUND, 0.0) / D["ground_h"])
        az_now = gain * ge * thrust_now - G + self.gust
        az_next = gain * ge * thrust_next - G + self.gust
        vz_new = self.vz + 0.5 * (az_now + az_next) * TS          # trapèzes sur [t_k, t_k+1]
        self.z += 0.5 * (self.vz + vz_new) * TS
        self.vz = vz_new
        self.az_prev, self.u_prev = az_now, u                     # = UnitDelay de az(t_k) et de u_k
        self.k += 1

        m = self._measure()
        obs = self.obs_builder(m)                                 # UN appel par pas

        # >>> IMPORTANT : REWARD (calculée sur l'état vrai, non bruité)
        e = m["z_ref"] - self.z
        d_sol = max(0.0, Z_SAFE - self.z) / 0.05                  # profondeur sous Z_SAFE (5 cm = 1)
        reward = (np.exp(-(e / E_TRACK) ** 2)                     # large : ramène vers la consigne
            + W_PREC * np.exp(-(e / E_PREC) ** 2)           # étroite : va chercher les derniers cm
            - W_V * min(self.vz ** 2, V2_MAX)
            - W_DU * (u - u_old) ** 2
            - W_SOL * min(d_sol, SOL_MAX))                # sous le sol
        terminated = not (Z_MIN*1.1 < self.z < 1.1*Z_MAX)
        if terminated:
            reward -= R_WORST * (N_STEPS - self.k + 1)            # mourir ne doit jamais être rentable
        truncated = self.k >= N_STEPS
        return obs, float(reward), bool(terminated), bool(truncated), {"m": m}

## 5. Outils : rollout, PID de référence, tracés, check MATLAB rejoué en Python

In [ ]:
def rollout(env, policy, seed=0):
    # policy(obs, m) -> u. Renvoie les signaux d'un épisode
    obs, info = env.reset(seed=seed)
    L = dict(t=[], z=[], z_ref=[], u=[], r=[])
    done = False
    while not done:
        m = info["m"]; u = float(policy(obs, m))
        obs, r, term, trunc, info = env.step(np.array([u], dtype=np.float32))
        L["t"].append(m["t"]); L["z_ref"].append(m["z_ref"]); L["u"].append(u); L["z"].append(env.z); L["r"].append(r)
        done = term or trunc
    return {k: np.array(v) for k, v in L.items()}

def rmse(L):
    return float(np.sqrt(np.mean((L["z_ref"] - L["z"]) ** 2)))

def summary(name, L):
    return (f"{name:8s} RMSE {rmse(L):.3f} m | min z {L['z'].min():+.3f} m | "
            f"sum du² {np.sum(np.diff(L['u'])**2):.2f} | retour {L['r'].sum():.0f}")

def make_pid(u_hover, kp=0.8, ki=0.1, kd=0.4):      # >>> gains réglés par grille : à revérifier sur ton RNN
    # u = U_HOVER + Kp e + Ki ∫e - Kd vz (dérivée sur la mesure : pas de pic aux marches)
    st = {"ie": 0.0}
    def pid(obs, m):
        if m["t"] == 0: st["ie"] = 0.0
        e = m["z_ref"] - m["z"]; st["ie"] = float(np.clip(st["ie"] + e * TS, -1, 1))
        return float(np.clip(u_hover + kp * e + ki * st["ie"] - kd * m["vz"], -1, 1))
    return pid

def plot_logs(logs, title=""):
    fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
    ax[0].step(logs[0][1]["t"], logs[0][1]["z_ref"], "k--", where="post", label="z_ref")
    for name, L in logs:
        ax[0].plot(L["t"], L["z"], label=f"{name} (RMSE {rmse(L):.3f})")
        ax[1].step(L["t"], L["u"], where="post", label=name)
    ax[0].axhline(Z_GROUND, c="k", ls=":"); ax[0].set_ylabel("z [m]"); ax[0].legend(fontsize=8)
    ax[1].set_ylabel("u"); ax[1].set_xlabel("t [s]"); ax[1].legend(fontsize=8)
    for a in ax: a.grid()
    ax[0].set_title(title); plt.tight_layout(); plt.show()

def check_like_matlab(policy_path="policy.mat", traj_path="traj_model.mat"):
    # Ce que fait check_controller dans EINT_RL_eval.m, avec les mêmes tolérances
    P = loadmat(policy_path, squeeze_me=True); T = loadmat(traj_path, squeeze_me=True)
    acts = str(P["activations"]).split(",")
    def forward(obs):
        x = np.asarray(obs, float).reshape(-1, 1)
        for i in range(1, int(P["n_layers"]) + 1):
            x = np.atleast_2d(P[f"W{i}"]) @ x + np.asarray(P[f"b{i}"], float).reshape(-1, 1)
            x = np.maximum(x, 0) if acts[i-1] == "relu" else np.tanh(x) if acts[i-1] == "tanh" else x
        lo, hi = np.atleast_1d(P["action_low"]), np.atleast_1d(P["action_high"]); x = x.ravel()
        out = str(P["output"])
        x = lo + 0.5 * (np.tanh(x) + 1) * (hi - lo) if out == "tanh_scale" else \
            lo + 0.5 * (x + 1) * (hi - lo) if out == "scale" else np.clip(x, lo, hi)
        return float(x[0])
    ob = ObservationBuilder(); ob.reset()                 # = clear rl_controller
    for k in range(len(T["t"])):
        m = {n: float(T[n][k]) for n in ("t", "z", "vz", "az_prev", "u_prev")}
        m["z_ref"] = mission(m["t"])
        assert abs(m["z_ref"] - T["z_ref"][k]) <= 1e-9, f"pas {k} : mission() diffère"
        obs = ob(m).astype(float)
        rel = np.max(np.abs(obs - T["obs"][k]) / np.maximum(1, np.abs(T["obs"][k])))
        assert rel <= 1e-5, f"pas {k} : composante {np.argmax(np.abs(obs - T['obs'][k]))+1} de l'observation diffère"
        u = forward(obs)
        assert abs(u - T["u"][k]) <= 1e-4, f"pas {k} : u = {u:.5f} vs {T['u'][k]:.5f} : réexporter"
    print(f"Check (Python) OK : {len(T['t'])} pas. Lance maintenant EINT_RL_eval dans MATLAB.")

## 6. Tests de l'environnement (à faire une fois, avant d'entraîner)

Dans l'ordre :

1. **Le RNN numpy reproduit ton RNN PyTorch** : compare avec `model(...)` dans `checkdata`, voir la cellule
   ci-dessous.
2. **Réponses libres** : à `u` constant, az doit se stabiliser (≈ −9,8 à `u = 0`, ≈ 0 vers `u ≈ 0,38`), et le tracé
   doit ressembler à ton essai Simulink.
3. **Poussée d'équilibre** `U_HOVER` : c'est ce que la politique doit apprendre à fournir au décollage.
4. **`check_env`**, **déterminisme** de l'env d'évaluation, actions aléatoires sans NaN.
5. **PID** : s'il suit la mission, l'env est sain. C'est aussi ta baseline pour le rapport.

In [ ]:
if globals().get("RUN_ENV_TESTS", True):
    from stable_baselines3.common.env_checker import check_env

    plant = RNNPlant()
    print(f"RNN : H = {len(plant.b)}, scaler sur {len(plant.mm_scale)} instant(s), préchauffage {plant.warmup} pas")

    # (1) À exécuter dans checkdata.ipynb pour comparer au réseau PyTorch :
    #   u = np.r_[np.zeros(40), np.full(60, 0.4), np.full(60, -0.3)]
    #   with torch.no_grad(): yn = model(((torch.tensor(u, dtype=torch.float32) - x_mean) / x_std)[None])[0].numpy()
    #   y = yn * float(y_std) + float(y_mean); az_torch = (y - sc.min_[:len(u)]) / sc.scale_[:len(u)]
    #   puis comparer à RNNPlant("rnn_plant.npz", warmup=0).free_response(u)[1]  (écart attendu ~1e-5)

    # (2) réponses libres après préchauffage
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    for u in [-0.5, 0.0, 0.3, 0.38, 0.45, 0.8]:
        Z, A = plant.free_response(np.full(40, u))
        tt = np.arange(40) * TS
        ax[0].plot(tt, Z, label=f"u={u}"); ax[1].plot(tt, A)
    ax[0].set_ylabel("z [m]"); ax[1].set_ylabel("az [m/s²]")
    for a in ax: a.grid(); a.set_xlabel("t depuis le départ [s]")
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

    # (3) poussée d'équilibre
    U_HOVER = plant.hover_command()
    print(f"poussée d'équilibre U_HOVER = {U_HOVER:.3f}")

    # (4) API, déterminisme, robustesse numérique
    check_env(DroneEnv(noise=1.0, randomize=True), warn=True)
    a = rollout(DroneEnv(), lambda o, m: U_HOVER); b = rollout(DroneEnv(), lambda o, m: U_HOVER)
    assert np.allclose(a["z"], b["z"]), "env d'évaluation non déterministe"
    rng = np.random.default_rng(0)
    for s in range(5):
        L = rollout(DroneEnv(noise=1.0, randomize=True), lambda o, m: rng.uniform(-1, 1), seed=s)
        assert np.all(np.isfinite(L["z"]))
    print("env OK")

    # (5) PID de référence
    L_pid = rollout(DroneEnv(), make_pid(U_HOVER))
    print(summary("PID", L_pid))
    plot_logs([("PID", L_pid)], "PID sur le RNN (mission nominale)")